<a href="https://www.kaggle.com/code/avikdas567/predicting-heart-disease?scriptVersionId=300688737" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostClassifier
import lightgbm as lgb

# Load Data

train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test  = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")

TARGET = "Heart Disease"
ID_COL = "id"

X = train.drop(columns=[TARGET])
y = train[TARGET].map({"Absence": 0, "Presence": 1}).astype(int)

X_test = test.copy()

# Encode Categorical Features

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

X_lgb = X.copy()
X_test_lgb = X_test.copy()

for col in cat_cols:
    le = LabelEncoder()
    full = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(full)
    X_lgb[col] = le.transform(X[col].astype(str))
    X_test_lgb[col] = le.transform(X_test[col].astype(str))

# Settings

N_SPLITS = 10
SEEDS = [42, 2024]

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

print("Training started...\n")

for seed in SEEDS:
    
    print(f"\n====== SEED {seed} ======\n")
    
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    
    oof_seed = np.zeros(len(X))
    test_seed = np.zeros(len(X_test))
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        
        print(f"Fold {fold}")
        
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        X_tr_lgb = X_lgb.iloc[tr_idx]
        X_val_lgb = X_lgb.iloc[val_idx]
        
        # CatBoost
        cat_model = CatBoostClassifier(
            iterations=4500,
            learning_rate=0.02,
            depth=8,
            l2_leaf_reg=10,
            border_count=128,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            task_type="GPU",
            early_stopping_rounds=250,
            verbose=False
        )
        
        cat_model.fit(
            X_tr,
            y_tr,
            eval_set=(X_val, y_val),
            cat_features=cat_cols,
            use_best_model=True
        )
        
        val_cat = cat_model.predict_proba(X_val)[:, 1]
        test_cat = cat_model.predict_proba(X_test)[:, 1]
        
        # LightGBM
        lgb_params = {
            "objective": "binary",
            "metric": "auc",
            "learning_rate": 0.02,
            "num_leaves": 96,
            "min_data_in_leaf": 40,
            "feature_fraction": 0.85,
            "bagging_fraction": 0.85,
            "bagging_freq": 5,
            "lambda_l1": 1.0,
            "lambda_l2": 3.0,
            "device": "gpu",
            "seed": seed,
            "verbosity": -1
        }
        
        lgb_train = lgb.Dataset(X_tr_lgb, y_tr)
        lgb_valid = lgb.Dataset(X_val_lgb, y_val)
        
        lgb_model = lgb.train(
            lgb_params,
            lgb_train,
            num_boost_round=6000,
            valid_sets=[lgb_valid],
            callbacks=[
                lgb.early_stopping(300),
                lgb.log_evaluation(0)
            ]
        )
        
        val_lgb = lgb_model.predict(X_val_lgb)
        test_lgb = lgb_model.predict(X_test_lgb)
        
        val_pred = 0.7 * val_cat + 0.3 * val_lgb
        test_pred = 0.7 * test_cat + 0.3 * test_lgb
        
        oof_seed[val_idx] = val_pred
        test_seed += test_pred / N_SPLITS
        
        fold_auc = roc_auc_score(y_val, val_pred)
        print(f"  AUC: {fold_auc:.6f}")
    
    seed_auc = roc_auc_score(y, oof_seed)
    print(f"\nSeed {seed} CV AUC: {seed_auc:.6f}")
    
    oof_preds += oof_seed / len(SEEDS)
    test_preds += test_seed / len(SEEDS)

# Final CV

cv_auc = roc_auc_score(y, oof_preds)
print("\n============================")
print(f"FINAL CV ROC-AUC: {cv_auc:.6f}")
print("============================\n")

# Submission

submission = pd.DataFrame({
    "id": test[ID_COL],
    "Heart Disease": test_preds
})

submission.to_csv("submission.csv", index=False)
print("submission.csv saved successfully.")
display(submission.head())

Training started...


====== SEED 42 ======

Fold 1


Default metric period is 5 because AUC is/are not implemented for GPU
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[993]	valid_0's auc: 0.955029
  AUC: 0.955122
Fold 2


Default metric period is 5 because AUC is/are not implemented for GPU


Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[810]	valid_0's auc: 0.955892
  AUC: 0.956041
Fold 3


...

Default metric period is 5 because AUC is/are not implemented for GPU


Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[675]	valid_0's auc: 0.954717
  AUC: 0.954842
Fold 10


Default metric period is 5 because AUC is/are not implemented for GPU


Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[829]	valid_0's auc: 0.954864
  AUC: 0.954939

Seed 2024 CV AUC: 0.955256

FINAL CV ROC-AUC: 0.955276

submission.csv saved successfully.


,id,Heart Disease
0,630000,0.943818
1,630001,0.006908
2,630002,0.986927
3,630003,0.003248
4,630004,0.175680
